[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanikina/low-resource-nlp-lab/blob/main/notebooks/PEFT_Tutorial.ipynb)

## 🤗 PEFT
PEFT responde a las siglas de Parameter-Efficient Fine-Tuning (Ajuste Fino Eficiente en Parámetros). La [Librería PEFT](https://github.com/huggingface/peft) admite diferentes métodos de adaptación para PLMs (Modelos de Lenguaje Preentrenados) mediante el ajuste fino de solo un pequeño número de parámetros, en lugar de actualizar todos los parámetros del modelo, lo que disminuye los costos computacionales y de almacenamiento. Por ejemplo, prompt tuning, LoRA e IA3 son compatibles con `PEFT` y además está integrada con las librerías `Transformers` y `Accelerate` para soportar entrenamiento distribuido e inferencia en modelos de gran tamaño.

Puedes encontrar tutoriales y ejemplos adicionales utilizando PEFT aquí: [https://huggingface.co/docs/peft/index](https://huggingface.co/docs/peft/index)

### 📦 Paso 1: Instalación e Importación de Librerías
En esta sección, se instalan e importan todas las dependencias necesarias:
- `transformers` y `peft`: Para cargar el modelo preentrenado y aplicar técnicas de ajuste fino eficiente.
- `datasets` y `evaluate`: De Hugging Face, para descargar, manejar datasets y evaluar el rendimiento del modelo.
- `torch` (PyTorch): El framework principal de Deep Learning sobre el que se entrenará el modelo.

In [13]:
!pip install transformers[torch]
!pip install datasets
!pip install peft
!pip install evaluate
!pip install accelerate
!pip install bitsandbytes
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 18.3 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import datasets
from transformers import AutoTokenizer

import torch
from torch import nn

import matplotlib.pyplot as plt

### ⚙️ Paso 2: Configuración del Entorno y Parámetros
Aquí configuramos el hardware y los hiperparámetros básicos:
- Detectamos si hay una **GPU (CUDA)** disponible para acelerar el entrenamiento, de lo contrario usaremos la CPU.
- Definimos el modelo base (`xlm-roberta-base`), que es un modelo multilíngüe basado en RoBERTa.
- Definimos hiperparámetros iniciales como el tamaño de lote (`batch_size`) y el número de épocas de entrenamiento (`num_epochs`).

### 🗃️ Preparación del Dataset
El dataset **Robot-Assisted Disaster Response** (Respuesta ante Desastres Asistida por Robots) consiste en conversaciones registradas durante sesiones de entrenamiento en el dominio de respuesta a emergencias. Las conversaciones ocurren típicamente entre varios operadores que controlan los robots, un líder de equipo y un comandante de misión. Cada turno de diálogo está anotado con una de las siguientes etiquetas de intención (intent labels):

| etiqueta | significado | train | porcentaje | ejemplo |
| --- | --- | --- | --- | --- |
| 0 | disconfirm (desconfirmar) | 35 | 1.3% | `Ist negativ, noch nicht.` |
| | | | | Is negative, not yet. (Es negativo, aún no) |
| 1 | order (orden/instrucción) | 216 | 8.3% | `Für Sie Erkundungsauftrag: Gesamtüberblick über die Einsatzstelle. Kommen.` |
| | | | | For you, reconnaissance assignment: overall overview of the site. Come in. (Para usted, misión de reconocimiento: visión general del lugar. Adelante.) |
| 2 | info_provide (proveer información) | 979 | 37.5% | `Ich verlasse das Erdgeschoss und gehe ins erste Obergeschoss.` |
| | | | | I leave the ground floor and go to the first floor. (Dejo la planta baja y voy al primer piso.) |
| 3 | info_request (solicitar información) | 238 | 9.1% | `Frage: Erkundungsergebnis aus der östlichen Seite des Gebäudes, kommen.` |
| | | | | Question: Exploration results from the eastern side of the building, come in. (Pregunta: Resultado de la exploración del lado este del edificio, adelante.) |
| 4 | call (llamada) | 487 | 18.7% | `RobLW an Zugführer, kommen.` |
| | | | | RobLW to platoon commander, come in. (RobLW a comandante de sección, adelante.) |
| 5 | call_response (respuesta a llamada) | 370 | 14.2% | `Ja, hier ist Zugführer, kommen.` |
| | | | | Yes, here is the platoon commander, come in. (Sí, aquí el comandante de sección, adelante.) |
| 6 | other (otro) | 43 | 1.7% | `Einen Augenblick, ich melde mich gleich.` |
| | | | | Just a moment, I will let you know soon. (Un momento, me comunico pronto.) |
| 7 | confirm (confirmar) | 242 | 9.3% | `Ein Lagebild von oben, komplette Lage, und ein Lagebild zwischen den beiden Türen, verstanden.` |
| | | | | A picture from above, complete setting, and a picture between both doors, understood. (Una imagen desde arriba, situación completa, y una imagen entre ambas puertas, comprendido.) |

### 🛠️ Paso 3: Preprocesamiento de Datos
Los modelos de Deep Learning no entienden texto directamente, necesitan números. En este paso:
1. Descargamos el dataset `DFKI/radr_intents` desde Hugging Face.
2. Concatenamos el rol de quien habla (`speaker`) con el mensaje (`text`) para dar más contexto al modelo.
3. Usamos el **Tokenizador** del modelo para convertir las palabras en representaciones numéricas (`input_ids` y `attention_mask`).
4. Renombramos la columna de objetivos a `labels` y eliminamos columnas de texto plano que ya no son necesarias.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Defining the task and hyperparameters
task = "intent_classification"
model_name = "xlm-roberta-base"
batch_size = 16
num_epochs = 20
encode_prev_turn = False

In [5]:
# Combine the speaker information with the input text
def add_speaker(example):
    example["text"] = example["speaker"] + " - " + example["text"]
    return example

In [6]:
# Defining the tokenizer and pre-processing the data
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

data = datasets.load_dataset("DFKI/radr_intents")
# Preparing the training data
train_task_dataset = train_task_dataset = data["train"] # datasets.Dataset.from_csv("radr_intents/train.csv")
train_task_dataset = train_task_dataset.map(add_speaker)
train_task_dataset = train_task_dataset.map(lambda samples: tokenizer(samples["text"]), batched=True)
train_task_dataset = train_task_dataset.rename_column("label","labels")
train_task_dataset = train_task_dataset.remove_columns(['id', 'speaker', 'text'])

# Preparing the validation data
dev_task_dataset = dev_task_dataset = data["validation"] # datasets.Dataset.from_csv("radr_intents/dev.csv")
dev_task_dataset = dev_task_dataset.map(add_speaker)
dev_task_dataset = dev_task_dataset.map(lambda samples: tokenizer(samples["text"]), batched=True)
dev_task_dataset = dev_task_dataset.rename_column("label","labels")
dev_task_dataset = dev_task_dataset.remove_columns(['id', 'speaker', 'text'])

# Printing some examples
for sample_i, sample in enumerate(dev_task_dataset):
    if sample_i > 2:
        break
    print(sample)
    print(tokenizer.batch_decode([sample["input_ids"][:30]], skip_special_tokens=True))

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/137k [00:00<?, ?B/s]

... (6 outputs left out)

Map:   0%|          | 0/2610 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]

Map:   0%|          | 0/310 [00:00<?, ? examples/s]

{'labels': 2, 'input_ids': [0, 345, 27816, 20, 345, 27816, 1256, 23752, 38953, 13, 4, 1439, 108879, 198, 404, 186, 169846, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['UAV - UAV hat Softwareprobleme, wir versuchen es zu beheben.']
{'labels': 2, 'input_ids': [0, 20602, 20, 823, 4, 493, 43254, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
['TL - Ja, verstanden.']
{'labels': 1, 'input_ids': [0, 20602, 20, 40787, 111697, 4, 2964, 4077, 599, 38250, 87523, 5, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['TL - Bitte melden, wenn wieder einsatzbereit.']


### ⚙️ Preparación del Modelo

### 🛡️ Paso 4: Configuración de PEFT (LoRA o P-Tuning)
En lugar de entrenar los millones de parámetros de `xlm-roberta-base` (lo cual requiere mucha memoria y tiempo), utilizamos **PEFT**.
- Aquí configuramos **LoRA (Low-Rank Adaptation)**, que congela los pesos originales del modelo e introduce matrices de bajo rango en las capas de atención (`query`, `key`, `value`).
- Esto reduce drásticamente el número de parámetros a entrenar a menos del 1%, manteniendo una precisión excelente.

In [9]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

from peft import (
    get_peft_config,
    get_peft_model,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
    PeftType,
    PromptEncoderConfig,
    LoraConfig,
)

import evaluate
import torch

task = "radr_intents"
num_epochs = 20
lr = 1e-3
batch_size = 16

Aquí podemos elegir entre `PromptEncoderConfig` para P-tuning, introducido en [GPT Understands, Too (Liu et al., 2021)](https://www.semanticscholar.org/paper/GPT-Understands%2C-Too-Liu-Zheng/bc37c6bdb8f39929a58b30464f72d6aa46cddc17), y `LoraConfig` para [LoRA: Low-Rank Adaptation of Large Language Models (Hu et al., 2021)](https://www.semanticscholar.org/paper/LoRA%3A-Low-Rank-Adaptation-of-Large-Language-Models-Hu-Shen/a8ca46b171467ceb2d7652fbfb67fe701ad86092).

In [10]:
peft_config = PromptEncoderConfig(task_type="SEQ_CLS", num_virtual_tokens=30, encoder_hidden_size=128)

In [11]:
peft_config = LoraConfig(
            r=8,
            lora_alpha=16,
            bias="none",
            task_type="SEQ_CLS",
            target_modules=["key", "query", "value"],
        )

### 🤖 Paso 5: Inicialización del Modelo con PEFT
En esta celda:
1. Cargamos el modelo base para clasificación de secuencias (`AutoModelForSequenceClassification`) con 8 clases posibles (las intenciones del dataset).
2. Envolvemos este modelo con nuestra configuración de LoRA (`get_peft_model`).
3. Imprimimos los parámetros entrenables. Verás que pasamos de entrenar unos 279 millones de parámetros a entrenar solo 1 millón aproximadamente (un **0.37%** del total).

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, return_dict=True)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
W1124 13:08:01.977000 966 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum sub

trainable params: 1,039,112 || all params: 279,088,912 || trainable%: 0.3723


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

### 📈 Paso 6: Configuración del Entrenamiento
Utilizamos la clase `Trainer` de Hugging Face, que simplifica el bucle de entrenamiento:
- Definimos los argumentos de entrenamiento (`TrainingArguments`), tales como la tasa de aprendizaje (`learning_rate`), dónde guardar los checkpoints, y la estrategia de evaluación por cada época.
- El `DataCollatorWithPadding` se encarga de agrupar dinámicamente las secuencias en lotes completando con relleno (*padding*) hasta la longitud de la secuencia más larga de cada lote.

### 🚀 Entrenamiento

### 🔥 Paso 7: Ejecución del Entrenamiento
Llamamos a `trainer.train()`. En este punto, el modelo comienza a procesar los datos por lotes (batches), calcula la pérdida (loss) y ajusta únicamente los parámetros que definimos en la configuración de LoRA a lo largo de 20 épocas.

In [ ]:
training_args = TrainingArguments(
    output_dir="roberta-base-peft",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    full_determinism=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_task_dataset,
    eval_dataset=dev_task_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss
1,No log,1.769225
2,No log,1.761213
3,No log,1.809234
4,1.780500,1.827829
5,1.780500,1.549241
6,1.780500,1.346341
7,1.523500,1.107880
8,1.523500,1.147788
9,1.523500,1.077515
10,1.196300,1.099513


TrainOutput(global_step=3280, training_loss=1.2007528537657204, metrics={'train_runtime': 210.9168, 'train_samples_per_second': 247.491, 'train_steps_per_second': 15.551, 'total_flos': 885640102603776.0, 'train_loss': 1.2007528537657204, 'epoch': 20.0})

### ✅ Evaluación

### 🎯 Paso 8: Inferencia y Prueba del Modelo
Una vez entrenado, probamos el modelo con oraciones completamente nuevas y simuladas para verificar su comportamiento:
- Pasamos las oraciones por el tokenizador.
- Las enviamos al modelo dentro de un bloque `torch.no_grad()` (ya que no necesitamos calcular gradientes en la fase de test).
- El modelo genera unas puntuaciones (*logits*), de las cuales tomamos la más alta (`argmax`) para mapearla a la categoría de intención correspondiente (`info_request`, `call`, etc.).

In [ ]:
id2label = {0: 'disconfirm', 1: 'order', 2: 'info_provide', 3: 'info_request', 4: 'call', 5: 'call_response', 6: 'other', 7: 'confirm'}
texts = ["UAV - UAV für Teamleader", "UGV 2 - Wir haben einen Kanister im Ergeschoss gefunden", "TL - Was ist deine aktuelle Position?"]
inputs = tokenizer(texts, truncation=True, padding="longest", return_tensors="pt")
inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs).logits
    print(outputs)
    labels = [id2label[label] for label in torch.argmax(outputs, dim=-1).tolist()]
    for text, label in zip(texts, labels):
        print(text, ">>>", label)

tensor([[-2.5557, -1.8112, -2.0377, -0.0248,  7.3617,  0.9375, -0.8120, -2.7155],
        [ 0.2110,  1.4479,  3.4419,  3.2218, -5.0600, -3.8530, -0.5526,  0.1037],
        [-0.8891,  1.6071,  1.1785,  5.2296, -3.2375, -3.1483, -0.1494, -1.5106]],
       device='cuda:0')
UAV - UAV für Teamleader >>> call
UGV 2 - Wir haben einen Kanister im Ergeschoss gefunden >>> info_provide
TL - Was ist deine aktuelle Position? >>> info_request
